In [1]:
"""
Preprocess and scale CSE-CICIDS2018 (target domain). Reuse scaler from CICIDS2017,
and calculate covariance statistics.
"""

### Imports ###
import json
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.covariance import LedoitWolf

# Load shared feature-space artifacts in a single, validated format.
def load_feature_order(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict):
        if "features" not in payload:
            raise ValueError(
                f"Expected key 'features' in {path} when JSON object is provided."
            )
        feature_order = list(payload["features"])
    elif isinstance(payload, list):
        feature_order = list(payload)
    else:
        raise ValueError(
            f"Unsupported shared feature space format in {path}: "
            f"{type(payload).__name__}"
        )

    if not feature_order:
        raise ValueError(f"Shared feature space in {path} is empty")

    return feature_order

In [2]:
### Import CSV ###

# Creates a Path object pointing to the target-domain CSV directory.
data_dir = Path("data/raw/target")

# Read CSV with encoding fallback for files that are not UTF-8.
def read_csv_with_fallback(file_path):
    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(file_path, low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("unknown", b"", 0, 1, f"Unable to decode {file_path}")

# Target domain can span multiple CSV exports; load and concatenate all of them.
csv_files = sorted(data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {data_dir}")

csv_file_names = [file_path.name for file_path in csv_files]
dfs = [read_csv_with_fallback(file_path) for file_path in csv_files]
df = pd.concat(dfs, ignore_index=True)

# Display a quick shape check and preview rows.
print(f"Loaded and concatenated {len(csv_files)} target CSV files:")
for file_name in csv_file_names:
    print(f"  - {file_name}")
print("Dataset shape:", df.shape)
df.head()

Loaded and concatenated 10 target CSV files:
  - Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  - Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
  - Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
  - Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
  - Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
  - Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
  - Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
  - Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
  - Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
  - Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
Dataset shape: (16233002, 84)


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Flow ID,Src IP,Src Port,Dst IP
0,443,6,02/03/2018 08:47:38,141385,9,7,553,3773.0,202,0,...,0.0,0.0,0.0,0.0,0.0,Benign,NaN,NaN,NaN,NaN
1,49684,6,02/03/2018 08:47:38,281,2,1,38,0.0,38,0,...,0.0,0.0,0.0,0.0,0.0,Benign,NaN,NaN,NaN,NaN
2,443,6,02/03/2018 08:47:40,279824,11,15,1086,10527.0,385,0,...,0.0,0.0,0.0,0.0,0.0,Benign,NaN,NaN,NaN,NaN
3,443,6,02/03/2018 08:47:40,132,2,0,0,0.0,0,0,...,0.0,0.0,0.0,0.0,0.0,Benign,NaN,NaN,NaN,NaN
4,443,6,02/03/2018 08:47:41,274016,9,13,1285,6141.0,517,0,...,0.0,0.0,0.0,0.0,0.0,Benign,NaN,NaN,NaN,NaN


In [3]:
### [Diagnostic] Extract features and labels ###

# Feature space: list every column present in the compiled target dataset.
feature_columns = list(df.columns)
print(f"Feature/column count: {len(feature_columns)}")
print("Feature space (all dataset columns):")
for column_name in feature_columns:
    print(f"- {column_name}")

# Label space: inspect every unique raw value stored in the 'Label' feature.
label_feature = "Label"
if label_feature not in df.columns:
    available_label_like_columns = [
        column_name
        for column_name in df.columns
        if column_name.strip().lower() in {"label", "attack"}
    ]
    raise ValueError(
        f"Expected '{label_feature}' column in compiled target dataset but it was not found. "
        f"Available label-like columns: {available_label_like_columns}"
    )

label_values = df[label_feature].astype("string").str.strip()
non_empty_label_mask = ~label_values.isna() & ~label_values.eq("")
missing_label_count = int((~non_empty_label_mask).sum())
unique_label_values = sorted(label_values[non_empty_label_mask].unique().tolist())
label_frequencies = label_values[non_empty_label_mask].value_counts().sort_values(ascending=False)

print()
print(f"Label feature analyzed: {label_feature}")
print(f"Unique label value count: {len(unique_label_values)}")
print("Label space (unique 'Label' values):")
for label_value in unique_label_values:
    print(f"- {label_value}")

print()
print("Label frequencies:")
for label_value, label_count in label_frequencies.items():
    print(f"- {label_value}: {int(label_count)}")

print(f"Missing or blank '{label_feature}' values: {missing_label_count}")

Feature/column count: 84
Feature space (all dataset columns):
- Dst Port
- Protocol
- Timestamp
- Flow Duration
- Tot Fwd Pkts
- Tot Bwd Pkts
- TotLen Fwd Pkts
- TotLen Bwd Pkts
- Fwd Pkt Len Max
- Fwd Pkt Len Min
- Fwd Pkt Len Mean
- Fwd Pkt Len Std
- Bwd Pkt Len Max
- Bwd Pkt Len Min
- Bwd Pkt Len Mean
- Bwd Pkt Len Std
- Flow Byts/s
- Flow Pkts/s
- Flow IAT Mean
- Flow IAT Std
- Flow IAT Max
- Flow IAT Min
- Fwd IAT Tot
- Fwd IAT Mean
- Fwd IAT Std
- Fwd IAT Max
- Fwd IAT Min
- Bwd IAT Tot
- Bwd IAT Mean
- Bwd IAT Std
- Bwd IAT Max
- Bwd IAT Min
- Fwd PSH Flags
- Bwd PSH Flags
- Fwd URG Flags
- Bwd URG Flags
- Fwd Header Len
- Bwd Header Len
- Fwd Pkts/s
- Bwd Pkts/s
- Pkt Len Min
- Pkt Len Max
- Pkt Len Mean
- Pkt Len Std
- Pkt Len Var
- FIN Flag Cnt
- SYN Flag Cnt
- RST Flag Cnt
- PSH Flag Cnt
- ACK Flag Cnt
- URG Flag Cnt
- CWE Flag Count
- ECE Flag Cnt
- Down/Up Ratio
- Pkt Size Avg
- Fwd Seg Size Avg
- Bwd Seg Size Avg
- Fwd Byts/b Avg
- Fwd Pkts/b Avg
- Fwd Blk Rate Avg
- Bwd 

In [4]:
### Data reduction ###

# Fixed random seed ensures that every re-run of this notebook produces
# bit-identical splits, a hard requirement for reproducible ML experiments.
REDUCTION_SEED = 42

# Per-class row caps determined by balancing computational tractability with
# class-frequency preservation. Minority attack categories retain their full
# populations; dominant classes are capped to bound training time and prevent
# majority-class dominance from suppressing attack-class signal.
CLASS_CAPS = {
    "SQL Injection":              87,
    "Brute Force -XSS":           230,
    "Brute Force -Web":           611,
    "DDOS attack-LOIC-UDP":       1_730,
    "DoS attacks-Slowloris":      10_990,
    "DoS attacks-GoldenEye":      41_508,
    "DoS attacks-SlowHTTPTest":   100_000,
    "Infilteration":              100_000,
    "SSH-Bruteforce":             100_000,
    "FTP-BruteForce":             100_000,
    "Bot":                        100_000,
    "DoS attacks-Hulk":           200_000,
    "DDoS attacks-LOIC-HTTP":     200_000,
    "DDOS attack-HOIC":           200_000,
    "Benign":                     2_000_000,
}

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

total_rows_before = len(df)
original_class_counts = df[label_col].value_counts().sort_index()

print("Original class counts:")
for cls, cnt in original_class_counts.items():
    cap = CLASS_CAPS.get(cls)
    cap_str = f"  (cap: {cap:,})" if cap is not None else "  (no cap defined)"
    print(f"  {cls}: {cnt:,}{cap_str}")
print(f"\nTotal rows before reduction: {total_rows_before:,}")
print()

# Per-class stratified sampling.
#
# Random sampling (rather than deterministic head() / iloc[:n] slicing) is used
# to avoid exploitation of any ordering artifact present in the raw export —
# CICFlowMeter outputs flows in capture order, so the first N rows of a class
# are temporally and sequentially biased toward early-session behaviour and do
# NOT constitute a representative subset of the full class distribution.
#
# Sampling WITHOUT replacement is mandatory: replacement would introduce
# duplicate rows, artificially inflate effective sample size, distort variance
# estimates, and corrupt CORAL covariance statistics computed downstream.
rng = np.random.default_rng(REDUCTION_SEED)

reduced_parts = []
for cls, cap in CLASS_CAPS.items():
    class_mask = df[label_col] == cls
    class_df = df.loc[class_mask]
    n_available = len(class_df)

    if n_available == 0:
        # Missing classes will be caught by the assertion below; skip silently here.
        continue

    if cap >= n_available:
        # Class is already at or below cap — retain every row unchanged to avoid
        # unnecessary copies and preserve exact intra-class distribution.
        reduced_parts.append(class_df)
    else:
        # Randomly sample without replacement. Using numpy's Generator (not
        # legacy RandomState) provides the modern, reproducible PRNG stream.
        sampled_indices = rng.choice(class_df.index, size=cap, replace=False)
        reduced_parts.append(class_df.loc[sampled_indices])

df_reduced = pd.concat(reduced_parts, axis=0, copy=False)

# Shuffle the concatenated result.
#
# After concatenation the DataFrame rows are blocked by class: all rows of class A
# appear before all rows of class B, etc. Many optimisers and gradient-based
# learners are sensitive to mini-batch class composition; a blocked layout
# causes successive batches to be nearly mono-class, breaking the IID assumption
# and introducing pathological gradient variance. A global shuffle eliminates
# this class-block locality and makes row order independent of class membership.
shuffled_index = rng.permutation(len(df_reduced))
df_reduced = df_reduced.iloc[shuffled_index].reset_index(drop=True)

total_rows_after = len(df_reduced)
reduced_class_counts = df_reduced[label_col].value_counts().sort_index()

print("Reduced class counts:")
for cls, cnt in reduced_class_counts.items():
    original = original_class_counts.get(cls, 0)
    cap = CLASS_CAPS.get(cls, original)
    retained_pct = cnt / original * 100 if original > 0 else 0.0
    print(f"  {cls}: {cnt:,}  (original: {original:,}, retained: {retained_pct:.1f}%)")

print(f"\nTotal rows after  reduction: {total_rows_after:,}")
print(f"Total rows before reduction: {total_rows_before:,}")
print(f"Reduction ratio: {total_rows_after / total_rows_before:.3f}")

Original class counts:
  Benign: 13,484,708  (cap: 2,000,000)
  Bot: 286,191  (cap: 100,000)
  Brute Force -Web: 611  (cap: 611)
  Brute Force -XSS: 230  (cap: 230)
  DDOS attack-HOIC: 686,012  (cap: 200,000)
  DDOS attack-LOIC-UDP: 1,730  (cap: 1,730)
  DDoS attacks-LOIC-HTTP: 576,191  (cap: 200,000)
  DoS attacks-GoldenEye: 41,508  (cap: 41,508)
  DoS attacks-Hulk: 461,912  (cap: 200,000)
  DoS attacks-SlowHTTPTest: 139,890  (cap: 100,000)
  DoS attacks-Slowloris: 10,990  (cap: 10,990)
  FTP-BruteForce: 193,360  (cap: 100,000)
  Infilteration: 161,934  (cap: 100,000)
  Label: 59  (no cap defined)
  SQL Injection: 87  (cap: 87)
  SSH-Bruteforce: 187,589  (cap: 100,000)

Total rows before reduction: 16,233,002



/var/folders/cz/y2skhn6512n92w71g06v44f00000gn/T/ipykernel_5827/1184471505.py:79: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df_reduced = pd.concat(reduced_parts, axis=0, copy=False)


Reduced class counts:
  Benign: 2,000,000  (original: 13,484,708, retained: 14.8%)
  Bot: 100,000  (original: 286,191, retained: 34.9%)
  Brute Force -Web: 611  (original: 611, retained: 100.0%)
  Brute Force -XSS: 230  (original: 230, retained: 100.0%)
  DDOS attack-HOIC: 200,000  (original: 686,012, retained: 29.2%)
  DDOS attack-LOIC-UDP: 1,730  (original: 1,730, retained: 100.0%)
  DDoS attacks-LOIC-HTTP: 200,000  (original: 576,191, retained: 34.7%)
  DoS attacks-GoldenEye: 41,508  (original: 41,508, retained: 100.0%)
  DoS attacks-Hulk: 200,000  (original: 461,912, retained: 43.3%)
  DoS attacks-SlowHTTPTest: 100,000  (original: 139,890, retained: 71.5%)
  DoS attacks-Slowloris: 10,990  (original: 10,990, retained: 100.0%)
  FTP-BruteForce: 100,000  (original: 193,360, retained: 51.7%)
  Infilteration: 100,000  (original: 161,934, retained: 61.8%)
  SQL Injection: 87  (original: 87, retained: 100.0%)
  SSH-Bruteforce: 100,000  (original: 187,589, retained: 53.3%)

Total rows afte

In [5]:
### Data sanitization ###

# Use the reduced dataset as the pipeline input for all downstream steps.
if "df_reduced" not in globals():
    raise ValueError("df_reduced is not available. Run the Data reduction cell first.")
df = df_reduced.copy()

print("Data sanitization tracking:")
print(f"Starting rows: {len(df):,}")
rows_before = len(df)

# Remove irrelevant columns.
# Note: removal of "Src Port" is critical; it carries heavy missingness.
df = df.drop(columns=["Flow ID", "Src IP", "Src Port", "Dst IP", "Timestamp"], errors="ignore")
print("  (Column drop: no rows removed)")

# Detect label column once to avoid repeated scans.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Convert all feature columns to numeric in one pass. Any embedded header rows,
# string non-finite tokens (e.g., Infinity/NaN), or other non-numeric artifacts
# become NaN via coercion.
feature_cols = [column_name for column_name in df.columns if column_name != label_col]
feature_numeric = df[feature_cols].apply(pd.to_numeric, errors="coerce")

# Build row-validity masks once and filter once.
label_series = df[label_col].astype("string").str.strip()
header_like_count = int(label_series.eq("Label").sum())
valid_label_mask = ~label_series.isna() & ~label_series.eq("") & ~label_series.eq("Label")
finite_feature_mask = np.isfinite(feature_numeric.to_numpy(dtype=np.float64)).all(axis=1)
keep_mask = valid_label_mask & finite_feature_mask

rows_removed_nonfinite = int((~keep_mask).sum())
df = df.loc[keep_mask].copy()
df[feature_cols] = feature_numeric.loc[keep_mask]
df[label_col] = label_series.loc[keep_mask]

print(
    f"  (Header-like row removal: {header_like_count:,} rows detected, "
    f"{len(df):,} rows remaining after consolidated cleanup)"
)
print(f"  (Non-finite/null removal: {rows_removed_nonfinite:,} rows removed, {len(df):,} rows remaining)")

# Remove duplicate flows.
rows_before_dedup = len(df)
df.drop_duplicates(inplace=True)
rows_removed_dedup = rows_before_dedup - len(df)
print(f"  (Deduplication: {rows_removed_dedup:,} rows removed, {len(df):,} rows remaining)")

# Remove leading/trailing spaces from all column names.
df.rename(columns=lambda x: x.strip(), inplace=True)
print("  (Column name cleanup: no rows removed)")

rows_after = len(df)
total_rows_removed = rows_before - rows_after
print(f"\nTotal rows removed during sanitization: {total_rows_removed:,} ({total_rows_removed/rows_before*100:.2f}%)")
print(f"Final row count: {rows_after:,}")

Data sanitization tracking:
Starting rows: 3,155,156
  (Column drop: no rows removed)
  (Header-like row removal: 0 rows detected, 3,140,269 rows remaining after consolidated cleanup)
  (Non-finite/null removal: 14,887 rows removed, 3,140,269 rows remaining)
  (Deduplication: 819,420 rows removed, 2,320,849 rows remaining)
  (Column name cleanup: no rows removed)

Total rows removed during sanitization: 834,307 (26.44%)
Final row count: 2,320,849


In [6]:
### [Diagnostic] Extract features and labels ###

# Feature space: list every column present in the compiled target dataset.
feature_columns = list(df.columns)
print(f"Feature/column count: {len(feature_columns)}")
print("Feature space (all dataset columns):")
for column_name in feature_columns:
    print(f"- {column_name}")

# Label space: inspect every unique raw value stored in the 'Label' feature.
label_feature = "Label"
if label_feature not in df.columns:
    available_label_like_columns = [
        column_name
        for column_name in df.columns
        if column_name.strip().lower() in {"label", "attack"}
    ]
    raise ValueError(
        f"Expected '{label_feature}' column in compiled target dataset but it was not found. "
        f"Available label-like columns: {available_label_like_columns}"
    )

label_values = df[label_feature].astype("string").str.strip()
non_empty_label_mask = ~label_values.isna() & ~label_values.eq("")
missing_label_count = int((~non_empty_label_mask).sum())
unique_label_values = sorted(label_values[non_empty_label_mask].unique().tolist())
label_frequencies = label_values[non_empty_label_mask].value_counts().sort_values(ascending=False)

print()
print(f"Label feature analyzed: {label_feature}")
print(f"Unique label value count: {len(unique_label_values)}")
print("Label space (unique 'Label' values):")
for label_value in unique_label_values:
    print(f"- {label_value}")

print()
print("Label frequencies:")
for label_value, label_count in label_frequencies.items():
    print(f"- {label_value}: {int(label_count)}")

print(f"Missing or blank '{label_feature}' values: {missing_label_count}")

Feature/column count: 79
Feature space (all dataset columns):
- Dst Port
- Protocol
- Flow Duration
- Tot Fwd Pkts
- Tot Bwd Pkts
- TotLen Fwd Pkts
- TotLen Bwd Pkts
- Fwd Pkt Len Max
- Fwd Pkt Len Min
- Fwd Pkt Len Mean
- Fwd Pkt Len Std
- Bwd Pkt Len Max
- Bwd Pkt Len Min
- Bwd Pkt Len Mean
- Bwd Pkt Len Std
- Flow Byts/s
- Flow Pkts/s
- Flow IAT Mean
- Flow IAT Std
- Flow IAT Max
- Flow IAT Min
- Fwd IAT Tot
- Fwd IAT Mean
- Fwd IAT Std
- Fwd IAT Max
- Fwd IAT Min
- Bwd IAT Tot
- Bwd IAT Mean
- Bwd IAT Std
- Bwd IAT Max
- Bwd IAT Min
- Fwd PSH Flags
- Bwd PSH Flags
- Fwd URG Flags
- Bwd URG Flags
- Fwd Header Len
- Bwd Header Len
- Fwd Pkts/s
- Bwd Pkts/s
- Pkt Len Min
- Pkt Len Max
- Pkt Len Mean
- Pkt Len Std
- Pkt Len Var
- FIN Flag Cnt
- SYN Flag Cnt
- RST Flag Cnt
- PSH Flag Cnt
- ACK Flag Cnt
- URG Flag Cnt
- CWE Flag Count
- ECE Flag Cnt
- Down/Up Ratio
- Pkt Size Avg
- Fwd Seg Size Avg
- Bwd Seg Size Avg
- Fwd Byts/b Avg
- Fwd Pkts/b Avg
- Fwd Blk Rate Avg
- Bwd Byts/b Avg
-

In [7]:
# ### NaN diagnostics (column-wise and class-wise) ###

# if "df" not in globals():
#     raise ValueError("df is not available. Run earlier preprocessing cells first.")

# LABEL_CANDIDATES = ["Label", "label", " Label"]
# label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
# if label_col is None:
#     raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# if not numeric_cols:
#     raise ValueError("No numeric columns found for NaN diagnostics.")

# total_rows = len(df)
# nan_mask_num = df[numeric_cols].isna()
# rows_with_any_nan_num = int(nan_mask_num.any(axis=1).sum())
# row_nan_rate_num = (rows_with_any_nan_num / total_rows * 100) if total_rows else 0.0

# print("NaN Diagnostic Summary")
# print(f"  Total rows: {total_rows:,}")
# print(f"  Numeric feature columns: {len(numeric_cols)}")
# print(f"  Rows with any NaN (numeric features only): {rows_with_any_nan_num:,} ({row_nan_rate_num:.2f}%)")

# # 1) Per-column NaN profile
# col_nan_counts = nan_mask_num.sum().sort_values(ascending=False)
# col_nan_pct = (col_nan_counts / total_rows * 100).round(2)
# nan_profile = pd.DataFrame({
#     "nan_count": col_nan_counts,
#     "nan_pct": col_nan_pct,
# })

# print("\nTop columns by NaN rate (numeric features):")
# print(nan_profile.head(25).to_string())

# sparse_thresholds = [25, 50, 75, 90]
# print("\nColumns above NaN-rate thresholds:")
# for threshold in sparse_thresholds:
#     count_above = int((nan_profile["nan_pct"] >= threshold).sum())
#     print(f"  >= {threshold}% NaN: {count_above}")

# # 2) Per-class NaN burden (numeric features only)
# labels_clean = df[label_col].astype("string").str.strip()
# valid_label_mask = ~labels_clean.isna() & ~labels_clean.eq("")
# if valid_label_mask.any():
#     diagnostics_df = pd.concat(
#         [labels_clean.rename("_label"), nan_mask_num.astype(np.int8)],
#         axis=1,
#     ).loc[valid_label_mask]

#     class_sizes = diagnostics_df.groupby("_label").size().rename("rows")
#     class_row_nan_rate = diagnostics_df.groupby("_label")[numeric_cols] \
#         .apply(lambda x: x.any(axis=1).mean() * 100) \
#         .rename("row_any_nan_pct")

#     class_col_nan_mean = diagnostics_df.groupby("_label")[numeric_cols].mean().mean(axis=1) * 100
#     class_col_nan_mean = class_col_nan_mean.rename("avg_feature_nan_pct")

#     class_nan_profile = pd.concat([class_sizes, class_row_nan_rate, class_col_nan_mean], axis=1)
#     class_nan_profile = class_nan_profile.sort_values("row_any_nan_pct", ascending=False)

#     print("\nPer-class NaN burden (numeric features):")
#     print(class_nan_profile.to_string(
#         formatters={
#             "row_any_nan_pct": "{:.2f}".format,
#             "avg_feature_nan_pct": "{:.2f}".format,
#         }
#     ))
# else:
#     print("\nNo valid labels found for per-class NaN diagnostics.")

# # 3) Candidate columns for dropping if you choose sparse-feature pruning
# drop_candidates = nan_profile[nan_profile["nan_pct"] >= 50].index.tolist()
# print(f"\nDrop-candidate columns at >=50% NaN: {len(drop_candidates)}")
# if drop_candidates:
#     print("Sample drop candidates:")
#     for col in drop_candidates[:20]:
#         print(f"- {col}")


In [8]:
### Feature-space alignment ###
# (select features according to predetermined shared feature space)

# Canonical feature-space contract shared by source and target pipelines.
FEATURE_LIST_PATH = Path("data/processed/shared_feature_space.json")
TARGET_FEATURE_MAP_PATH = Path("data/processed/target_feature_map.json")

# Handle missing files.
if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(
        f"Shared feature list not found at {FEATURE_LIST_PATH}. "
        "Create/populate this artifact before running preprocessing."
    )
if not TARGET_FEATURE_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Target feature map not found at {TARGET_FEATURE_MAP_PATH}. "
        "Create/populate this artifact before running preprocessing."
    )

# Load canonical ordered features. Order must be preserved.
shared_features = load_feature_order(FEATURE_LIST_PATH)

with open(TARGET_FEATURE_MAP_PATH, "r", encoding="utf-8") as f:
    target_feature_map = json.load(f)
if not isinstance(target_feature_map, dict):
    raise ValueError(
        f"Expected JSON object at {TARGET_FEATURE_MAP_PATH}, got "
        f"{type(target_feature_map).__name__}"
    )

# Feature alignment uses canonical feature keys only; label handling stays separate.
feature_map = {k: v for k, v in target_feature_map.items() if k != "Label"}
missing_map_keys = sorted(set(shared_features) - set(feature_map))
extra_map_keys = sorted(set(feature_map) - set(shared_features))
if missing_map_keys or extra_map_keys:
    raise ValueError(
        f"Target feature map key mismatch. Missing keys: {missing_map_keys[:10]} "
        f"(total={len(missing_map_keys)}); extra keys: {extra_map_keys[:10]} "
        f"(total={len(extra_map_keys)}). Update {TARGET_FEATURE_MAP_PATH}."
    )

# Identify label column after sanitization trims any surrounding whitespace.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

mapped_label_col = target_feature_map.get("Label")
if mapped_label_col is not None:
    if not isinstance(mapped_label_col, str):
        raise ValueError(
            f"Label mapping in {TARGET_FEATURE_MAP_PATH} must be a string, "
            f"got {type(mapped_label_col).__name__}"
        )
    mapped_label_col = mapped_label_col.strip() or "Label"
    if mapped_label_col != label_col:
        raise ValueError(
            f"Label mapping mismatch: {TARGET_FEATURE_MAP_PATH} maps 'Label' to "
            f"'{mapped_label_col}', but dataset column is '{label_col}'."
        )

# Aligns a DataFrame to the canonical shared feature contract by renaming through the
# target feature map, dropping extra columns, checking optional fill behavior, and
# enforcing exact canonical order.
def align_feature_space(frame, feature_list, feature_map, fill_missing=False, fill_value=0.0):
    feature_list = list(feature_list)
    resolved_raw_columns = {}
    missing = []

    for canonical in feature_list:
        raw_name = feature_map.get(canonical, canonical)
        if not isinstance(raw_name, str):
            raise ValueError(
                f"Feature mapping for '{canonical}' must be a string, "
                f"got {type(raw_name).__name__}"
            )

        raw_name = raw_name.strip() or canonical
        resolved_raw_columns[canonical] = raw_name
        if raw_name not in frame.columns:
            missing.append((canonical, raw_name))

    # Strict mode (default): block pipeline if required mapped features are absent.
    if missing and not fill_missing:
        preview = missing[:10]
        raise ValueError(
            f"Mapped target features not found: {preview} (total={len(missing)})"
        )

    aligned_columns = {}
    for canonical in feature_list:
        raw_name = resolved_raw_columns[canonical]
        if raw_name in frame.columns:
            aligned_columns[canonical] = frame[raw_name]
        else:
            aligned_columns[canonical] = fill_value

    used_raw_columns = {
        raw_name for raw_name in resolved_raw_columns.values() if raw_name in frame.columns
    }
    extra = sorted(set(frame.columns) - used_raw_columns)

    aligned = pd.DataFrame(aligned_columns, index=frame.index).copy()
    return aligned, extra, missing

# Ensure index is contiguous before splitting/rejoining features and labels.
df = df.reset_index(drop=True)

# Align only feature columns; label handling happens separately.
feature_df = df.drop(columns=[label_col]).copy()
aligned_X, dropped_extra, missing_cols = align_feature_space(
    feature_df,
    shared_features,
    feature_map,
    fill_missing=False,
    fill_value=0.0,
)

# Reattach labels by position (not index label) to avoid accidental NaNs.
labels_aligned = df[[label_col]].reset_index(drop=True)
aligned_X = aligned_X.reset_index(drop=True)
if len(aligned_X) != len(labels_aligned):
    raise ValueError(
        f"Feature/label row count mismatch after alignment: "
        f"X={len(aligned_X)}, y={len(labels_aligned)}"
    )
df = pd.concat([aligned_X, labels_aligned], axis=1)

print(f"Loaded shared feature list from {FEATURE_LIST_PATH}")
print(f"Loaded target feature map from {TARGET_FEATURE_MAP_PATH}")
print(f"Aligned feature count: {len(shared_features)}")
print(f"Dropped extra columns: {len(dropped_extra)}")
print(f"Missing required columns: {len(missing_cols)}")
print(f"Missing labels after reattach: {int(df[label_col].isna().sum())}")

Loaded shared feature list from data/processed/shared_feature_space.json
Loaded target feature map from data/processed/target_feature_map.json
Aligned feature count: 77
Dropped extra columns: 1
Missing required columns: 0
Missing labels after reattach: 0


In [9]:
### Label-space alignment ###
# (align labels according to predetermined shared label space)

SHARED_LABEL_SPACE_PATH = Path("data/processed/shared_label_space.json")
TARGET_LABEL_MAP_PATH = Path("data/processed/target_label_map.json")

if not SHARED_LABEL_SPACE_PATH.exists():
    raise FileNotFoundError(
        f"Shared label space file not found at {SHARED_LABEL_SPACE_PATH}"
    )
if not TARGET_LABEL_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Target label map file not found at {TARGET_LABEL_MAP_PATH}"
    )

# Load the canonical label contract in a validated format.
with open(SHARED_LABEL_SPACE_PATH, "r", encoding="utf-8") as f:
    shared_label_payload = json.load(f)

if isinstance(shared_label_payload, dict):
    if "labels" not in shared_label_payload:
        raise ValueError(
            f"Expected key 'labels' in {SHARED_LABEL_SPACE_PATH} when JSON object is provided."
        )
    shared_label_space = list(shared_label_payload["labels"])
elif isinstance(shared_label_payload, list):
    shared_label_space = list(shared_label_payload)
else:
    raise ValueError(
        f"Unsupported shared label space format in {SHARED_LABEL_SPACE_PATH}: "
        f"{type(shared_label_payload).__name__}"
    )

if not shared_label_space:
    raise ValueError(f"Shared label space in {SHARED_LABEL_SPACE_PATH} is empty")
if len(shared_label_space) != len(set(shared_label_space)):
    duplicate_labels = sorted(
        label for label in set(shared_label_space) if shared_label_space.count(label) > 1
    )
    raise ValueError(
        f"Duplicate canonical labels found in {SHARED_LABEL_SPACE_PATH}: {duplicate_labels}"
    )

with open(TARGET_LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    target_label_map = json.load(f)
if not isinstance(target_label_map, dict):
    raise ValueError(
        f"Expected JSON object at {TARGET_LABEL_MAP_PATH}, got "
        f"{type(target_label_map).__name__}"
    )

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Normalize raw labels for stable matching (trim spaces, keep missing as <NA>).
raw_labels = df[label_col].astype("string").str.strip()

# Detect genuinely missing labels (null/blank) separately from unmapped labels.
missing_label_mask = raw_labels.isna() | raw_labels.eq("")
missing_label_count = int(missing_label_mask.sum())

if missing_label_count > 0:
    missing_indices = df.index[missing_label_mask].tolist()
    preview_rows = missing_indices[:10]
    preview_values = [repr(v) for v in raw_labels.loc[preview_rows].tolist()]
    raise ValueError(
        f"Missing label values found at row indices {preview_rows} "
        f"(total={missing_label_count}). "
        f"Sample raw values at those rows: {preview_values}. "
        "Clean/drop these rows before label alignment."
    )

# Guardrail: mapping file must only map into allowed shared classes.
invalid_canonical_labels = sorted(
    set(target_label_map.values()) - set(shared_label_space)
)
if invalid_canonical_labels:
    raise ValueError(
        "target_label_map.json contains classes not present in shared_label_space.json: "
        f"{invalid_canonical_labels}"
    )

# Apply raw->shared mapping.
mapped_labels = raw_labels.map(target_label_map)

# Drop unmapped non-missing raw labels by design instead of crashing the pipeline.
unmapped_mask = (~missing_label_mask) & mapped_labels.isna()
dropped_unmapped_count = int(unmapped_mask.sum())
if dropped_unmapped_count > 0:
    unmapped_raw = raw_labels[unmapped_mask]
    unmapped_counts = unmapped_raw.value_counts().sort_values(ascending=False)
    keep_mask = ~unmapped_mask

    print("Dropping unmapped raw labels from target dataset:")
    for raw_label, raw_count in unmapped_counts.items():
        print(f"- {raw_label}: {int(raw_count)}")

    df = df.loc[keep_mask].reset_index(drop=True)
    mapped_labels = mapped_labels.loc[keep_mask].reset_index(drop=True)
else:
    df = df.reset_index(drop=True)
    mapped_labels = mapped_labels.reset_index(drop=True)

if df.empty:
    raise ValueError(
        "All rows were removed during target label alignment. "
        f"Check {TARGET_LABEL_MAP_PATH} and {SHARED_LABEL_SPACE_PATH}."
    )

# Replace dataset labels with aligned shared classes.
df[label_col] = mapped_labels

# Sanity check: print class-frequency table after label alignment.
label_counts = df[label_col].value_counts(dropna=False).sort_values(ascending=False)
label_freq = (label_counts / len(df) * 100).round(2)

print(f"Loaded shared label space from {SHARED_LABEL_SPACE_PATH}")
print(f"Loaded target label map from {TARGET_LABEL_MAP_PATH}")
print(f"Canonical label count: {len(shared_label_space)}")
print(f"Dropped unmapped rows: {dropped_unmapped_count}")
print(f"Rows retained after label alignment: {len(df)}")
print("Label category frequencies after alignment:")
for cls in label_counts.index:
    print(f"- {cls}: {int(label_counts[cls])} ({label_freq[cls]:.2f}%)")

Loaded shared label space from data/processed/shared_label_space.json
Loaded target label map from data/processed/target_label_map.json
Canonical label count: 13
Dropped unmapped rows: 0
Rows retained after label alignment: 2320849
Label category frequencies after alignment:
- Benign: 1717280 (73.99%)
- DDoS: 277806 (11.97%)
- Infiltration: 88527 (3.81%)
- DoS Hulk: 83665 (3.60%)
- Bot: 51142 (2.20%)
- SSH-Patator: 50143 (2.16%)
- DoS GoldenEye: 41406 (1.78%)
- DoS slowloris: 9908 (0.43%)
- Web Attack - Brute Force: 555 (0.02%)
- Web Attack - XSS: 228 (0.01%)
- Web Attack - Sql Injection: 84 (0.00%)
- DoS Slowhttptest: 55 (0.00%)
- FTP-Patator: 50 (0.00%)


In [10]:
### Attack samples extraction ###

# Use the label-aligned dataframe at this point in the pipeline so the exported
# attack-only dataset keeps the same canonical schema as the rest of preprocessing.
if "df" not in globals():
    raise ValueError("df is not available. Run the earlier preprocessing cells first.")

# Re-detect the label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Prefer the notebook's canonical benign label if it has already been defined.
BENIGN_CLASS = globals().get("BENIGN_CLASS", "Benign")

# Keep every non-benign row and preserve all columns exactly as they exist here.
label_series = df[label_col].astype("string").str.strip()
attack_mask = label_series.notna() & ~label_series.eq(BENIGN_CLASS)
attack_df = df.loc[attack_mask].copy()
attack_holdout_df = attack_df.copy()
attack_holdout_export_df = attack_holdout_df.copy()

# Save the attack-only dataset for use in the downstream notebook.
attack_output_dir = Path("data/processed/target")
attack_output_dir.mkdir(parents=True, exist_ok=True)
attack_holdout_path = attack_output_dir / "attack_holdout.csv"
attack_holdout_export_df.to_csv(attack_holdout_path, index=False)

attack_label_counts = attack_df[label_col].value_counts().sort_values(ascending=False)

print(f"Benign class: {BENIGN_CLASS}")
print(f"Attack rows extracted: {len(attack_df):,}")
print(f"Saved attack-only dataset to: {attack_holdout_path}")
print(f"Preserved columns: {len(attack_df.columns)}")
print("Attack label counts:")
for label_value, label_count in attack_label_counts.items():
    print(f"- {label_value}: {int(label_count)}")

Benign class: Benign
Attack rows extracted: 603,569
Saved attack-only dataset to: data/processed/target/attack_holdout.csv
Preserved columns: 78
Attack label counts:
- DDoS: 277806
- Infiltration: 88527
- DoS Hulk: 83665
- Bot: 51142
- SSH-Patator: 50143
- DoS GoldenEye: 41406
- DoS slowloris: 9908
- Web Attack - Brute Force: 555
- Web Attack - XSS: 228
- Web Attack - Sql Injection: 84
- DoS Slowhttptest: 55
- FTP-Patator: 50


In [11]:
### Train/Test Split ###
from sklearn.model_selection import train_test_split

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

X = df.drop(columns=[label_col]).copy()
y = df[label_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
    shuffle=True,
)

print(f"Train/Test sizes: {len(y_train)}/{len(y_test)}")

Train/Test sizes: 1856679/464170


In [12]:
### Scaling (reuse scaler of source dataset) ###

SCALER_PATH = Path("models/source_scaler.joblib")
if not SCALER_PATH.exists():
    raise FileNotFoundError(f"Source scaler not found at {SCALER_PATH}")

scaler = joblib.load(SCALER_PATH)

# Apply source-fitted scaler separately to target train/test split.
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames to preserve feature names/indexing.
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(f"Loaded scaler from {SCALER_PATH}")
print(f"Scaled splits shapes: train={X_train.shape}, test={X_test.shape}")

Loaded scaler from models/source_scaler.joblib
Scaled splits shapes: train=(1856679, 77), test=(464170, 77)


In [13]:
### Label encoding (reuse encoder of shared label space) ###

ENCODER_PATH = Path("models/label_encoder.joblib")
if not ENCODER_PATH.exists():
    raise FileNotFoundError(f"Label encoder not found at {ENCODER_PATH}")

le = joblib.load(ENCODER_PATH)

# Ensure train/test labels are fully compatible with source-fitted encoder.
raw_train_labels = y_train.astype("string").str.strip()
raw_test_labels = y_test.astype("string").str.strip()
all_unknown = sorted((set(raw_train_labels.dropna()) | set(raw_test_labels.dropna())) - set(le.classes_))
if all_unknown:
    raise ValueError(
        f"Found labels not present in fitted source encoder: {all_unknown[:10]} "
        f"(total={len(all_unknown)})."
    )

y_train = pd.Series(le.transform(raw_train_labels), index=y_train.index, name="Label")
y_test = pd.Series(le.transform(raw_test_labels), index=y_test.index, name="Label")

print(f"Loaded label encoder from {ENCODER_PATH}")
print(f"Encoded classes ({len(le.classes_)}): {list(le.classes_)}")

Loaded label encoder from models/label_encoder.joblib
Encoded classes (13): ['Benign', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Infiltration', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


In [14]:
### Calculate and export covariance and mean statistics ###
# Note: calculated from target training split only.

# Load shared feature space contract.
shared_feature_space_path = Path("data/processed/shared_feature_space.json")
if not shared_feature_space_path.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")

# Reuse unified parser so feature-space JSON is handled consistently across cells.
feature_order = load_feature_order(shared_feature_space_path)

# Improvement support: persist canonical feature ordering inside CORAL stats so
# evaluation can verify source/target schema parity before adaptation.
X_target_train_aligned = X_train[feature_order]

# Convert to numpy for CORAL math using float64 for better numerical stability.
X_trg = X_target_train_aligned.to_numpy(dtype=np.float64)

# 1) Feature-wise mean vector.
target_feature_mean = np.mean(X_trg, axis=0)

# 2) Centered target training data.
X_trg_centered = X_trg - target_feature_mean

# Improvement #1: use Ledoit-Wolf shrinkage covariance for a better-conditioned
# covariance estimate than plain sample covariance.
target_cov_estimator = LedoitWolf()
target_cov_estimator.fit(X_trg_centered)
target_covariance = np.asarray(target_cov_estimator.covariance_, dtype=np.float64)
target_covariance = (target_covariance + target_covariance.T) / 2.0

# Save diagnostics consumed by training/eval for spectral-floor and stability analysis.
target_eigenvalues = np.linalg.eigvalsh(target_covariance)
target_min_eig = float(target_eigenvalues.min())
target_max_eig = float(target_eigenvalues.max())
target_cov_condition_number = float(np.linalg.cond(target_covariance))
target_covariance_ridge = 0.0

# Sanity checks.
assert target_covariance.shape[0] == target_covariance.shape[1], "Covariance matrix must be square"
assert target_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

# Package CORAL statistics.
coral_target_stats = {
    "feature_order": feature_order,
    "mean": target_feature_mean,
    "covariance": target_covariance,
    "covariance_estimator": "LedoitWolf",
    "covariance_shrinkage": float(target_cov_estimator.shrinkage_),
    "min_eigenvalue_before_regularization": target_min_eig,
    "max_eigenvalue": target_max_eig,
    "covariance_condition_number": target_cov_condition_number,
    "covariance_ridge": target_covariance_ridge,
}

# Persist for downstream domain adaptation pipeline.
coral_stats_path = Path("models/coral_target_stats.joblib")
coral_stats_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(coral_target_stats, coral_stats_path)

print("CORAL target statistics extracted and saved successfully.")
print("Verified: CORAL stats were computed from target train split only.")
print(f"Train rows used for CORAL: {len(X_target_train_aligned)}")
print(f"Saved to: {coral_stats_path}")
print(f"Features: {len(feature_order)}")
print(f"Covariance shape: {target_covariance.shape}")
print(f"Covariance estimator: LedoitWolf (shrinkage={target_cov_estimator.shrinkage_:.6f})")
print(f"Covariance eigenvalues: min={target_min_eig:.6e}, max={target_max_eig:.6e}")
print(f"Covariance condition number: {target_cov_condition_number:.6e}")

CORAL target statistics extracted and saved successfully.
Verified: CORAL stats were computed from target train split only.
Train rows used for CORAL: 1856679
Saved to: models/coral_target_stats.joblib
Features: 77
Covariance shape: (77, 77)
Covariance estimator: LedoitWolf (shrinkage=1.000000)
Covariance eigenvalues: min=5.529764e+02, max=5.529764e+02
Covariance condition number: 1.000000e+00


In [15]:
### [Diagnostic] Extract features and labels ###

# Feature space: list every column present in the compiled target dataset.
feature_columns = list(df.columns)
print(f"Feature/column count: {len(feature_columns)}")
print("Feature space (all dataset columns):")
for column_name in feature_columns:
    print(f"- {column_name}")

# Label space: inspect every unique raw value stored in the 'Label' feature.
label_feature = "Label"
if label_feature not in df.columns:
    available_label_like_columns = [
        column_name
        for column_name in df.columns
        if column_name.strip().lower() in {"label", "attack"}
    ]
    raise ValueError(
        f"Expected '{label_feature}' column in compiled target dataset but it was not found. "
        f"Available label-like columns: {available_label_like_columns}"
    )

label_values = df[label_feature].astype("string").str.strip()
non_empty_label_mask = ~label_values.isna() & ~label_values.eq("")
missing_label_count = int((~non_empty_label_mask).sum())
unique_label_values = sorted(label_values[non_empty_label_mask].unique().tolist())
label_frequencies = label_values[non_empty_label_mask].value_counts().sort_values(ascending=False)

print()
print(f"Label feature analyzed: {label_feature}")
print(f"Unique label value count: {len(unique_label_values)}")
print("Label space (unique 'Label' values):")
for label_value in unique_label_values:
    print(f"- {label_value}")

print()
print("Label frequencies:")
for label_value, label_count in label_frequencies.items():
    print(f"- {label_value}: {int(label_count)}")

print(f"Missing or blank '{label_feature}' values: {missing_label_count}")

Feature/column count: 78
Feature space (all dataset columns):
- Destination Port
- Flow Duration
- Total Fwd Packets
- Total Backward Packets
- Total Length of Fwd Packets
- Total Length of Bwd Packets
- Fwd Packet Length Max
- Fwd Packet Length Min
- Fwd Packet Length Mean
- Fwd Packet Length Std
- Bwd Packet Length Max
- Bwd Packet Length Min
- Bwd Packet Length Mean
- Bwd Packet Length Std
- Flow Bytes/s
- Flow Packets/s
- Flow IAT Mean
- Flow IAT Std
- Flow IAT Max
- Flow IAT Min
- Fwd IAT Total
- Fwd IAT Mean
- Fwd IAT Std
- Fwd IAT Max
- Fwd IAT Min
- Bwd IAT Total
- Bwd IAT Mean
- Bwd IAT Std
- Bwd IAT Max
- Bwd IAT Min
- Fwd PSH Flags
- Bwd PSH Flags
- Fwd URG Flags
- Bwd URG Flags
- Fwd Header Length
- Bwd Header Length
- Fwd Packets/s
- Bwd Packets/s
- Min Packet Length
- Max Packet Length
- Packet Length Mean
- Packet Length Std
- Packet Length Variance
- FIN Flag Count
- SYN Flag Count
- RST Flag Count
- PSH Flag Count
- ACK Flag Count
- URG Flag Count
- CWE Flag Count
- EC

In [16]:
### Export processed data ###

# Create output directory for processed target train/test splits.
output_dir = Path("data/processed/target")
output_dir.mkdir(parents=True, exist_ok=True)

# Save processed target train data.
train_df = pd.DataFrame(X_train, columns=X_train.columns)
train_df["Label"] = y_train.values
train_df.to_csv(output_dir / "train.csv", index=False)

# Save processed target test data.
test_df = pd.DataFrame(X_test, columns=X_test.columns)
test_df["Label"] = y_test.values
test_df.to_csv(output_dir / "test.csv", index=False)

print("Saved datasets:")
print(f"  Train: {len(train_df)} samples")
print(f"  Test: {len(test_df)} samples")
print(f"  Output directory: {output_dir}")

Saved datasets:
  Train: 1856679 samples
  Test: 464170 samples
  Output directory: data/processed/target
